# 07 · Kaggle Standalone Training Notebook (Bulletproof & Zero-Error)

This notebook reverse-engineers all dependencies used in **Notebook 03 (`03_train.ipynb`)** and runs training on **Kaggle (30 GB RAM + GPU)** without missing-file errors.

### What Notebook 03 Uses Behind the Scenes:
1. **AASIST Codebase:** `aasist/config/AASIST.conf`, `models/AASIST.py`, `evaluation.py`, patched for NumPy.
2. **ASVspoof 2019 LA Dataset:**
   - `ASVspoof2019_LA_train/flac/` (25,380 clean training clips)
   - `ASVspoof2019_LA_dev/flac/` (24,844 validation clips for dev EER scoring)
   - `ASVspoof2019_LA_cm_protocols/` (`train.trn.txt` & `dev.trl.txt`)
3. **Augmented Training Set (`train_aug`):**
   - `generated/train_aug/flac/` (7,614 noise/reverb-augmented clips)
   - `generated/train_aug/protocol_aug.txt` (augmented labels protocol)
4. **Fast Disk Working Directory:** Everything unpacked to `/kaggle/working/fast` on high-speed NVMe.

---
### Kaggle Settings Checklist:
- [x] **Accelerator:** GPU P100 or T4 (`Settings` → `Accelerator`)
- [x] **Internet:** On (`Settings` → `Internet`)
- [x] **Data attached:** Add your dataset(s) containing `.tar` files or ASVspoof data (`+ Add Data` on top right)

## Step 1 · Clone Repository & Setup Fast Disk

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/createsomethingmarvellous/mini-project.git'
BASE = Path('/kaggle/working/mini_project')
FAST = Path('/kaggle/working/fast')

BASE.mkdir(parents=True, exist_ok=True)
FAST.mkdir(parents=True, exist_ok=True)

# Set environment variables so all project scripts read from fast local SSD
os.environ['MP_BASE'] = str(BASE)
os.environ['MP_DATA'] = str(FAST)

if (BASE / '.git').exists():
    print('Repo exists. Pulling latest code...')
    !git -C {BASE} pull -q --rebase
else:
    print(f'Cloning repository into {BASE}...')
    !git clone -q {REPO_URL} {BASE}

os.chdir(BASE)
!pip install -q audiomentations speechbrain soundfile librosa tqdm
print('✓ Base environment and packages ready.')

## Step 2 · Auto-Detect All Input Datasets & Archives
This scans `/kaggle/input` recursively to find where your archives (`asvspoof.tar`, `train_aug.tar`, etc.) or uncompressed folders live.

In [ ]:
from pathlib import Path
import tarfile, zipfile, shutil

print('=== Scanning /kaggle/input ===')
input_root = Path('/kaggle/input')
found_tars = {}

if input_root.exists():
    for file_path in input_root.rglob('*'):
        if file_path.suffix in ['.tar', '.gz'] or file_path.name.endswith('.tar.gz'):
            stem = file_path.name.replace('.tar.gz', '').replace('.tar', '')
            found_tars[stem] = file_path
            print(f'  📦 Archive: {file_path.name:<25} ({file_path.stat().st_size / 1e9:.2f} GB) at {file_path}')

if not found_tars:
    print('  ⚠️ No .tar archives found in /kaggle/input.')
    print('  Checking if uncompressed folders already exist in /kaggle/input...')
    for folder in input_root.glob('*'):
        if folder.is_dir():
            print(f'  📁 Directory: {folder.name}')
else:
    print(f'✓ Found {len(found_tars)} archive(s).')

## Step 3 · Unpack Audio & Augmented Data to Fast SSD (`/kaggle/working/fast`)
Extracts `asvspoof` (clean audio) and `train_aug` (noise-augmented audio) needed for Model B training.

In [ ]:
os.chdir(BASE)

# 1. Unpack using fast_data.py
print('Running fast_data.py unpack...')
!python scripts/fast_data.py unpack asvspoof
!python scripts/fast_data.py unpack train_aug

# 2. Fallback direct extraction if fast_data missed any custom archive name
for key, tar_path in found_tars.items():
    if 'asvspoof' in key.lower() and not (FAST / 'asvspoof2019').exists():
        print(f'Extracting {tar_path.name} directly to {FAST}...')
        with tarfile.open(tar_path, 'r:*') as tar:
            tar.extractall(FAST)
    elif 'train_aug' in key.lower() and not (FAST / 'generated/train_aug').exists():
        print(f'Extracting {tar_path.name} directly to {FAST}...')
        with tarfile.open(tar_path, 'r:*') as tar:
            tar.extractall(FAST)

print('\nLocal Fast SSD Status:')
!python scripts/fast_data.py status

## Step 4 · Setup AASIST Codebase & NumPy Patch
Clones official AASIST repository into `mini_project/aasist` and patches deprecated `np.float` references.

In [ ]:
os.chdir(BASE)
aasist_dir = BASE / 'aasist'

if not (aasist_dir / 'main.py').exists():
    # Check if uploaded as a dataset
    aasist_inputs = [p for p in input_root.rglob('aasist') if (p / 'main.py').exists()]
    if aasist_inputs:
        print(f'Copying AASIST from {aasist_inputs[0]}...')
        shutil.copytree(aasist_inputs[0], aasist_dir, dirs_exist_ok=True)
    else:
        print('Cloning fresh AASIST repository from GitHub...')
        !git clone --depth=1 -q https://github.com/clovaai/aasist.git {aasist_dir}

print('Applying NumPy compatibility patch...')
!python scripts/setup_aasist.py
print('✓ AASIST code ready at', aasist_dir)

## Step 5 · The Pre-Flight Integrity Test (Zero-Error Guarantee)
This cell tests all 6 critical components required by `train_model.py`. **If anything is missing, it will tell you here before training starts.**

In [ ]:
import torch
os.chdir(BASE)

print('================ PRE-FLIGHT VERIFICATION ================')

# 1. GPU
has_gpu = torch.cuda.is_available()
print(f"[{'PASS' if has_gpu else 'FAIL'}] GPU Device        : {torch.cuda.get_device_name(0) if has_gpu else 'NO GPU FOUND'}")

# 2. AASIST files
has_aasist = (BASE / 'aasist/main.py').exists() and (BASE / 'aasist/config/AASIST.conf').exists()
print(f"[{'PASS' if has_aasist else 'FAIL'}] AASIST Code       : {BASE / 'aasist'}")

# 3. Protocols
sys.path.insert(0, str(BASE / 'scripts'))
import config, common

try:
    train_proto = common.protocol_path('train')
    dev_proto = common.protocol_path('dev')
    has_protocols = train_proto.exists() and dev_proto.exists()
    print(f"[{'PASS' if has_protocols else 'FAIL'}] ASVspoof Protocols : {train_proto.name}, {dev_proto.name}")
except Exception as e:
    has_protocols = False
    print(f"[FAIL] ASVspoof Protocols : Error locating protocols: {e}")

# 4. Audio training splits
try:
    train_flac_dir = common.split_dir('train') / 'flac'
    dev_flac_dir = common.split_dir('dev') / 'flac'
    n_train = len(list(train_flac_dir.glob('*.flac')))
    n_dev = len(list(dev_flac_dir.glob('*.flac')))
    has_audio = n_train > 0 and n_dev > 0
    print(f"[{'PASS' if has_audio else 'FAIL'}] Clean Audio Files  : Train={n_train} files, Dev={n_dev} files")
except Exception as e:
    has_audio = False
    print(f"[FAIL] Clean Audio Files  : Error locating splits: {e}")

# 5. Augmented training data (Required for Model B)
aug_proto = config.TRAIN_AUG_DIR / 'protocol_aug.txt'
aug_flac_dir = config.TRAIN_AUG_DIR / 'flac'
n_aug = len(list(aug_flac_dir.glob('*.flac'))) if aug_flac_dir.exists() else 0
has_aug = aug_proto.exists() and n_aug > 0
print(f"[{'PASS' if has_aug else 'FAIL'}] Augmented Training : {n_aug} files, protocol={aug_proto.exists()}")

print('=========================================================')
if has_gpu and has_aasist and has_protocols and has_audio and has_aug:
    print('🚀 ALL SYSTEMS GO! You are 100% ready to train Model B without errors.')
else:
    print('⚠️ Please review the failed checks above before proceeding.')

## Step 6 · Train Model B
Choose your target seed: `42`, `123`, or `2024`.  
Training automatically saves progress every 10 minutes and resumes seamlessly if interrupted.

In [ ]:
os.chdir(BASE)

# Select which seed to train:
SEED = 42   # Change to 123 or 2024 for the other two seeds

print(f'Starting Model B Training (Seed {SEED})...')
print(f'Config: Batch Size = {config.BATCH_SIZE}, Workers = {config.NUM_WORKERS}, High RAM = 30GB')
print('=' * 65)

!python scripts/train_model.py --tag B --seed {SEED}

## Step 7 · Bundle & Download Checkpoints
Creates a `.zip` archive of your trained checkpoints (`swa.pth`, `best_dev.pth`, `last_checkpoint.pth`)  
so you can download them directly from the **Output** panel in Kaggle's right sidebar.

In [ ]:
import zipfile
os.chdir(BASE)

ckpt_dir = BASE / f'checkpoints/B_seed{SEED}'
out_zip = Path(f'/kaggle/working/checkpoints_B_seed{SEED}.zip')

if ckpt_dir.exists():
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in ckpt_dir.rglob('*'):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(ckpt_dir).as_posix())
                print(f'  Zipped: {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
    print(f'\n✓ Checkpoints saved to: {out_zip} ({out_zip.stat().st_size / 1e6:.1f} MB)')
    print('👉 Look at the "Output" panel on the right sidebar to download your file.')
else:
    print('No checkpoints found yet.')